# 00 - Import CSV with All Data
**The CSV is expected to be in this format**
- ID and absolute paths to niftis are critical
```
+-----+----------------------------+--------------+--------------+--------------+
| ID  | Nifti_File_Path            | Covariate_1  | Covariate_2  | Covariate_3  |
+-----+----------------------------+--------------+--------------+--------------+
| 1   | /path/to/file1.nii.gz      | 0.5          | 1.2          | 3.4          |
| 2   | /path/to/file2.nii.gz      | 0.7          | 1.4          | 3.1          |
| 3   | /path/to/file3.nii.gz      | 0.6          | 1.5          | 3.5          |
| 4   | /path/to/file4.nii.gz      | 0.9          | 1.1          | 3.2          |
| ... | ...                        | ...          | ...          | ...          |
+-----+----------------------------+--------------+--------------+--------------+
```

Prep Output Direction

In [ ]:
# Specify where you want to save your results to
out_dir = "/Volumes/OneTouch/01p_Schmahmann_SCA_Atrophy/results/optimzation/symptom_on_lhs/parcelwise_spcorrel/motor-cog"

Import Data

In [ ]:
# Specify the path to your CSV file containing NIFTI paths
input_csv_path = '/Volumes/OneTouch/01p_Schmahmann_SCA_Atrophy/results/optimzation/optimized_master_list.csv'
sheet = None

In [ ]:
from calvin_utils.permutation_analysis_utils.statsmodels_palm import CalvinStatsmodelsPalm
# Instantiate the PalmPrepararation class
cal_palm = CalvinStatsmodelsPalm(input_csv_path=input_csv_path, output_dir=out_dir, sheet=sheet)
# Call the process_nifti_paths method
data_df = cal_palm.read_and_display_data()
data_df


# 01 - Preprocess Your Data

**Handle NANs**
- Set drop_nans=True is you would like to remove NaNs from data
- Provide a column name or a list of column names to remove NaNs from

In [ ]:
data_df.columns

In [ ]:
drop_list = ['selected_Nifti_File_Path']

In [ ]:
data_df = cal_palm.drop_nans_from_columns(columns_to_drop_from=drop_list)
data_df

**Drop Row Based on Value of Column**

Define the column, condition, and value for dropping rows
- column = 'your_column_name'
- condition = 'above'  # Options: 'equal', 'above', 'below'

In [ ]:
data_df.columns

Set the parameters for dropping rows

In [ ]:
column = 'selected'  # The column you'd like to evaluate
condition = 'not'  # The condition to check ('equal', 'above', 'below', 'not')
value = 1 # The value to drop if found

In [ ]:
data_df, other_df = cal_palm.drop_rows_based_on_value(column, condition, value)
display(data_df)

Standardize

In [ ]:
# Remove anything you don't want to standardize
cols_not_to_standardize = ['subid']

In [ ]:
# data_df = cal_palm.standardize_columns(cols_not_to_standardize)
data_df

Convert Categorical to Ordinal

One Hot Encode

In [ ]:
# import pandas as pd
# one_hot = pd.get_dummies(data_df['cluster_label'], prefix='cluster_label', dtype=int)
# data_df = data_df.join(one_hot)
# data_df

# 02 - Run Evaluation

In [ ]:
for col in data_df.columns:
    print(col)

In [ ]:
from calvin_utils.permutation_analysis_utils import ParcelwiseRegressionSimilarity
analysis = ParcelwiseRegressionSimilarity(
    df=data_df,
    formula_1="Nifti_File_Path ~ TotalCCASFailScore",
    formula_2="Nifti_File_Path ~ TotalBarsScore",
    contrast_matrix=[[1]],
    parcel_path="/Volumes/HowExp/resources/atlases/mni_space/aal_atlas/AAL_MNI_V7_fine_rois", #expects a folder
    parcel_file_pattern="*.nii.gz",
    mask_path="/Users/cu135/Software_Local/calvin_utils_project/circuit_pyper/resources/MNI152_T1_2mm_brain_mask.nii",
    voxelwise_vars=["Nifti_File_Path"],
    voxelwise_interactions=None,
    data_transform_method="standardize",
    regression_type="linear",
    n_permutations=1000,
    similarity="pearson",
    add_intercept=False
)

results = analysis.run(tails="two_tail", fwe=False)

In [ ]:
results